# 03 — Model Training & Evaluation
**Bank Retention Intelligence Platform**

This notebook covers:
- Loading features dataset from notebook 02
- Train/test split + SMOTE for class imbalance
- Training 7 ML models: Logistic Regression, Decision Tree, Random Forest, Extra Trees, XGBoost, LightGBM, CatBoost
- Model comparison and best model selection
- Confusion matrix, ROC curve, Feature Importance
- Saving best model to `models/best_model.pkl`

> **Input:** `data/processed/features_dataset.csv`  
> **Output:** `models/best_model.pkl`, `data/processed/model_comparison.csv`

In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, average_precision_score,
                             confusion_matrix, roc_curve)
from imblearn.over_sampling import SMOTE

from src.feature_engineering import get_feature_list
from src.visualization import (plot_model_comparison, plot_roc_curve,
                                plot_confusion_matrix, plot_feature_importance)
from src.utils import load_processed, save_processed, save_model, save_figure

print("Libraries loaded")

Libraries loaded


## 1. Load Features Dataset

In [2]:
df = load_processed('features_dataset.csv')
FEATURE_COLS = get_feature_list()
print(f"Dataset: {df.shape[0]:,} rows | Churn rate: {df['Exited'].mean()*100:.1f}%")
print(f"Features: {len(FEATURE_COLS)}")
df.head(3)

Dataset: 10,000 rows | Churn rate: 20.4%
Features: 15


,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,...,BalanceSalaryRatio,ProductsPerTenure,EngagementScore,WealthScore,RelationshipStrength,AgeGroup,WealthSegment,HighValueDisengaged,Geography_enc,Gender_enc
0,619,France,Female,42,2,0.00,1,1,1,101348.88,...,0.000000,0.333333,1.0,0.954465,0.46,36-45,Low,0,0,0
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,...,0.744670,0.500000,0.7,1.435852,0.31,36-45,Medium,0,1,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,...,1.401362,0.333333,0.3,1.597223,0.56,36-45,High,1,0,0


## 2. Train/Test Split

In [3]:
X = df[FEATURE_COLS].fillna(0)
y = df['Exited']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train size : {len(X_train):,}  |  Churn: {y_train.mean()*100:.1f}%")
print(f"Test size  : {len(X_test):,}   |  Churn: {y_test.mean()*100:.1f}%")

Train size : 8,000  |  Churn: 20.4%
Test size  : 2,000   |  Churn: 20.3%


## 3. Scaling + SMOTE

In [4]:
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

sm = SMOTE(random_state=42)
X_res, y_res = sm.fit_resample(X_train_sc, y_train)

print(f"Before SMOTE: {dict(y_train.value_counts())}")
print(f"After  SMOTE: {dict(pd.Series(y_res).value_counts())}")
print(f"Training samples after SMOTE: {len(X_res):,}")

Before SMOTE: {0: 6370, 1: 1630}
After  SMOTE: {1: 6370, 0: 6370}
Training samples after SMOTE: 12,740


## 4. Define Models

In [5]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree':       DecisionTreeClassifier(max_depth=8, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    'Extra Trees':         ExtraTreesClassifier(n_estimators=200, random_state=42, n_jobs=-1),
}

try:
    from xgboost import XGBClassifier
    models['XGBoost'] = XGBClassifier(n_estimators=300, learning_rate=0.05,
                                       max_depth=6, use_label_encoder=False,
                                       eval_metric='logloss', random_state=42)
    print("✓ XGBoost available")
except ImportError:
    print("✗ XGBoost not installed")

try:
    from lightgbm import LGBMClassifier
    models['LightGBM'] = LGBMClassifier(n_estimators=300, learning_rate=0.05,
                                         random_state=42, verbose=-1)
    print("✓ LightGBM available")
except ImportError:
    print("✗ LightGBM not installed")

try:
    from catboost import CatBoostClassifier
    models['CatBoost'] = CatBoostClassifier(iterations=500, learning_rate=0.05,
                                             depth=6, random_seed=42, verbose=0)
    print("✓ CatBoost available")
except ImportError:
    print("✗ CatBoost not installed")

print(f"\nTotal models to train: {len(models)}")

✓ XGBoost available
✓ LightGBM available
✓ CatBoost available

Total models to train: 7


## 5. Train All Models

In [6]:
results = []
trained_models = {}

for name, clf in models.items():
    print(f"Training {name:25s} ...", end=' ', flush=True)
    clf.fit(X_res, y_res)
    pred  = clf.predict(X_test_sc)
    proba = clf.predict_proba(X_test_sc)[:, 1]
    results.append({
        'Model':     name,
        'Accuracy':  round(accuracy_score(y_test, pred)  * 100, 2),
        'Precision': round(precision_score(y_test, pred) * 100, 2),
        'Recall':    round(recall_score(y_test, pred)    * 100, 2),
        'F1':        round(f1_score(y_test, pred), 4),
        'ROC_AUC':   round(roc_auc_score(y_test, proba)  * 100, 2),
        'PR_AUC':    round(average_precision_score(y_test, proba) * 100, 2),
    })
    trained_models[name] = (clf, proba)
    print(f"ROC-AUC={results[-1]['ROC_AUC']:.2f}%  Recall={results[-1]['Recall']:.2f}%")

results_df = pd.DataFrame(results).sort_values('ROC_AUC', ascending=False).reset_index(drop=True)
save_processed(results_df, 'model_comparison.csv')
print("\nModel comparison saved.")

Training Logistic Regression       ... 

ROC-AUC=77.55%  Recall=70.52%
Training Decision Tree             ... 

ROC-AUC=81.79%  Recall=70.27%


Training Random Forest             ... 

ROC-AUC=84.91%  Recall=57.00%
Training Extra Trees               ... 

ROC-AUC=83.51%  Recall=51.35%
Training XGBoost                   ... 

ROC-AUC=85.64%  Recall=57.25%
Training LightGBM                  ... 

ROC-AUC=85.80%  Recall=55.53%
Training CatBoost                  ... 

ROC-AUC=86.31%  Recall=54.79%
  Saved → E:\bank-retention-intelligence\data\processed\model_comparison.csv  (7 rows)

Model comparison saved.


## 6. Model Comparison Table

In [7]:
print("\n" + "="*70)
print(f"{'Model':25s} {'Accuracy':>9} {'Precision':>10} {'Recall':>8} {'F1':>7} {'ROC-AUC':>9}")
print("="*70)
for _, row in results_df.iterrows():
    marker = " ◀ BEST" if _ == 0 else ""
    print(f"{row['Model']:25s} {row['Accuracy']:>8.2f}% {row['Precision']:>9.2f}% "
          f"{row['Recall']:>7.2f}% {row['F1']:>6.4f} {row['ROC_AUC']:>8.2f}%{marker}")
print("="*70)


Model                      Accuracy  Precision   Recall      F1   ROC-AUC
CatBoost                     85.50%     67.78%   54.79% 0.6060    86.31% ◀ BEST
LightGBM                     85.55%     67.66%   55.53% 0.6100    85.80%
XGBoost                      85.65%     67.34%   57.25% 0.6189    85.64%
Random Forest                84.00%     61.54%   57.00% 0.5918    84.91%
Extra Trees                  83.65%     61.83%   51.35% 0.5611    83.51%
Decision Tree                78.10%     47.43%   70.27% 0.5663    81.79%
Logistic Regression          72.15%     39.64%   70.52% 0.5075    77.55%


## 7. Model Comparison Chart

In [8]:
fig = plot_model_comparison(results_df)
save_figure(fig, 'model_comparison.png')
plt.show()

  Figure saved → E:\bank-retention-intelligence\data\processed\figures\model_comparison.png


## 8. Best Model — Evaluation

In [9]:
best_name  = results_df.iloc[0]['Model']
best_clf, best_proba = trained_models[best_name]
best_pred  = best_clf.predict(X_test_sc)

print(f"Best Model: {best_name}")
print(f"  Accuracy : {accuracy_score(y_test, best_pred)*100:.2f}%")
print(f"  Precision: {precision_score(y_test, best_pred)*100:.2f}%")
print(f"  Recall   : {recall_score(y_test, best_pred)*100:.2f}%")
print(f"  F1 Score : {f1_score(y_test, best_pred):.4f}")
print(f"  ROC-AUC  : {roc_auc_score(y_test, best_proba)*100:.2f}%")
print(f"  PR-AUC   : {average_precision_score(y_test, best_proba)*100:.2f}%")

Best Model: CatBoost
  Accuracy : 85.50%
  Precision: 67.78%
  Recall   : 54.79%
  F1 Score : 0.6060
  ROC-AUC  : 86.31%
  PR-AUC   : 71.19%


## 9. Confusion Matrix

In [10]:
cm = confusion_matrix(y_test, best_pred)
print(f"\nConfusion Matrix:\n{cm}")
tn, fp, fn, tp = cm.ravel()
print(f"  True Negatives  (correctly retained): {tn:,}")
print(f"  True Positives  (correctly churned) : {tp:,}")
print(f"  False Positives (false alarm)        : {fp:,}")
print(f"  False Negatives (missed churners)    : {fn:,}")

fig = plot_confusion_matrix(cm)
save_figure(fig, 'confusion_matrix.png')
plt.show()


Confusion Matrix:
[[1487  106]
 [ 184  223]]


  True Negatives  (correctly retained): 1,487
  True Positives  (correctly churned) : 223
  False Positives (false alarm)        : 106
  False Negatives (missed churners)    : 184


  Figure saved → E:\bank-retention-intelligence\data\processed\figures\confusion_matrix.png


## 10. ROC Curve

In [11]:
from sklearn.metrics import roc_curve
fpr, tpr, _ = roc_curve(y_test, best_proba)
auc_val = roc_auc_score(y_test, best_proba)

fig = plot_roc_curve(fpr, tpr, auc_val)
save_figure(fig, 'roc_curve.png')
plt.show()

  Figure saved → E:\bank-retention-intelligence\data\processed\figures\roc_curve.png


## 11. Feature Importance

In [12]:
if hasattr(best_clf, 'feature_importances_'):
    imp = pd.DataFrame({
        'Feature':    FEATURE_COLS,
        'Importance': best_clf.feature_importances_
    }).sort_values('Importance', ascending=False)

    print("Top 10 most important features:")
    print(imp.head(10).to_string(index=False))

    save_processed(imp, 'feature_importance.csv')

    fig = plot_feature_importance(imp['Feature'].tolist(), imp['Importance'].tolist())
    save_figure(fig, 'feature_importance.png')
    plt.show()
else:
    print(f"{best_name} does not expose feature_importances_")

Top 10 most important features:
             Feature  Importance
              Tenure   30.153385
                 Age   16.201756
       Geography_enc   15.739332
       NumOfProducts   11.398357
   ProductsPerTenure    8.288945
             Balance    3.127292
RelationshipStrength    2.473345
          Gender_enc    2.393271
     EstimatedSalary    2.199492
         CreditScore    2.147541
  Saved → E:\bank-retention-intelligence\data\processed\feature_importance.csv  (15 rows)


  Figure saved → E:\bank-retention-intelligence\data\processed\figures\feature_importance.png


## 12. Save Best Model

In [13]:
save_model(best_clf, scaler, FEATURE_COLS, 'best_model.pkl')
print(f"\nModel bundle saved:")
print(f"  model    : {best_name}")
print(f"  scaler   : StandardScaler (fitted on {len(X_train):,} samples)")
print(f"  features : {len(FEATURE_COLS)} columns")
print("\nNext: Run 04_business_insights.ipynb")

  Model saved → E:\bank-retention-intelligence\models\best_model.pkl

Model bundle saved:
  model    : CatBoost
  scaler   : StandardScaler (fitted on 8,000 samples)
  features : 15 columns

Next: Run 04_business_insights.ipynb
